<a href="https://colab.research.google.com/github/sabyapaul/skills-introduction-to-github/blob/main/Agentic_AI_Bronze_to_Silver_Eligibility_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 1: Install PySpark**

In [1]:
!pip install pyspark -q

**Step 2: Start Spark Session**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp

spark = SparkSession.builder \
    .appName("Agentic_AI_Bronze_to_Silver_Eligibility_Validation") \
    .getOrCreate()

**Step 3: Create Sample Bronze Data**

**Bronze Claims**


In [3]:
claims_data = [
    ("C001", "M001", "2026-01-15", 1200.00),
    ("C002", "M002", "2026-02-10", 800.00),
    ("C003", "M003", "2026-03-05", 500.00),
    ("C004", "M004", "2026-01-20", 700.00),
    ("C005", "M001", "2026-04-10", 300.00)
]

bronze_claim = spark.createDataFrame(
    claims_data,
    ["claim_id", "member_id", "service_date", "claim_amount"]
)

bronze_claim = bronze_claim.withColumn(
    "service_date", col("service_date").cast("date")
)

bronze_claim.show()

+--------+---------+------------+------------+
|claim_id|member_id|service_date|claim_amount|
+--------+---------+------------+------------+
|    C001|     M001|  2026-01-15|      1200.0|
|    C002|     M002|  2026-02-10|       800.0|
|    C003|     M003|  2026-03-05|       500.0|
|    C004|     M004|  2026-01-20|       700.0|
|    C005|     M001|  2026-04-10|       300.0|
+--------+---------+------------+------------+



**Bronze Eligibility**

In [4]:
eligibility_data = [
    ("M001", "2026-01-01", "2026-03-31"),
    ("M002", "2026-01-01", "2026-12-31"),
    ("M003", "2026-04-01", "2026-12-31")
]

bronze_eligibility = spark.createDataFrame(
    eligibility_data,
    ["member_id", "effective_date", "termination_date"]
)

bronze_eligibility = bronze_eligibility \
    .withColumn("effective_date", col("effective_date").cast("date")) \
    .withColumn("termination_date", col("termination_date").cast("date"))

bronze_eligibility.show()

+---------+--------------+----------------+
|member_id|effective_date|termination_date|
+---------+--------------+----------------+
|     M001|    2026-01-01|      2026-03-31|
|     M002|    2026-01-01|      2026-12-31|
|     M003|    2026-04-01|      2026-12-31|
+---------+--------------+----------------+



**Step 4:Load Bronze to Silver**

In [5]:
silver_claim = bronze_claim
silver_eligibility = bronze_eligibility

silver_claim.createOrReplaceTempView("silver_claim")
silver_eligibility.createOrReplaceTempView("silver_eligibility")

**Step 5: Create Agent Action Function**

In [7]:
def flag_claims(invalid_df, reason_code):
    flagged_df = invalid_df \
        .withColumn("validation_status", lit("FAILED")) \
        .withColumn("reason_code", lit(reason_code)) \
        .withColumn("agent_action", lit("FLAG_FOR_REVIEW")) \
        .withColumn("created_timestamp", current_timestamp())

    return flagged_df

**Step 6: Build Eligibility Validation Agent**

In [8]:
class EligibilityValidationAgent:

    def __init__(self, spark):
        self.spark = spark
        self.agent_name = "Eligibility Validation Agent"

    def validate(self):
        print(f"Running {self.agent_name}...")

        invalid_eligibility = self.spark.sql("""
            SELECT c.claim_id, c.member_id, c.service_date, c.claim_amount
            FROM silver_claim c
            LEFT JOIN silver_eligibility e
              ON c.member_id = e.member_id
             AND c.service_date BETWEEN e.effective_date AND e.termination_date
            WHERE e.member_id IS NULL
        """)

        invalid_count = invalid_eligibility.count()

        if invalid_count > 0:
            print(f"{invalid_count} invalid claims found.")
            action_df = flag_claims(
                invalid_eligibility,
                "NO_ACTIVE_ELIGIBILITY"
            )
            return action_df
        else:
            print("All claims passed eligibility validation.")
            return None

**Step 7: Run the Agent**

In [9]:
eligibility_agent = EligibilityValidationAgent(spark)

flagged_claims = eligibility_agent.validate()

if flagged_claims:
    flagged_claims.show(truncate=False)

Running Eligibility Validation Agent...
3 invalid claims found.
+--------+---------+------------+------------+-----------------+---------------------+---------------+--------------------------+
|claim_id|member_id|service_date|claim_amount|validation_status|reason_code          |agent_action   |created_timestamp         |
+--------+---------+------------+------------+-----------------+---------------------+---------------+--------------------------+
|C003    |M003     |2026-03-05  |500.0       |FAILED           |NO_ACTIVE_ELIGIBILITY|FLAG_FOR_REVIEW|2026-06-10 02:40:00.880738|
|C004    |M004     |2026-01-20  |700.0       |FAILED           |NO_ACTIVE_ELIGIBILITY|FLAG_FOR_REVIEW|2026-06-10 02:40:00.880738|
|C005    |M001     |2026-04-10  |300.0       |FAILED           |NO_ACTIVE_ELIGIBILITY|FLAG_FOR_REVIEW|2026-06-10 02:40:00.880738|
+--------+---------+------------+------------+-----------------+---------------------+---------------+--------------------------+



**Step 8: Create Final Silver Claims Table**
Valid claims go forward. Invalid claims are flagged.

In [10]:
valid_claims = spark.sql("""
    SELECT c.*
    FROM silver_claim c
    INNER JOIN silver_eligibility e
      ON c.member_id = e.member_id
     AND c.service_date BETWEEN e.effective_date AND e.termination_date
""")

valid_claims = valid_claims \
    .withColumn("validation_status", lit("PASSED")) \
    .withColumn("reason_code", lit(None).cast("string"))

valid_claims.show()

+--------+---------+------------+------------+-----------------+-----------+
|claim_id|member_id|service_date|claim_amount|validation_status|reason_code|
+--------+---------+------------+------------+-----------------+-----------+
|    C001|     M001|  2026-01-15|      1200.0|           PASSED|       NULL|
|    C002|     M002|  2026-02-10|       800.0|           PASSED|       NULL|
+--------+---------+------------+------------+-----------------+-----------+



**Step 9: Combine Valid and Invalid Claims**

In [11]:
final_silver_claims = valid_claims.select(
    "claim_id", "member_id", "service_date", "claim_amount",
    "validation_status", "reason_code"
).unionByName(
    flagged_claims.select(
        "claim_id", "member_id", "service_date", "claim_amount",
        "validation_status", "reason_code"
    )
)

final_silver_claims.show(truncate=False)

+--------+---------+------------+------------+-----------------+---------------------+
|claim_id|member_id|service_date|claim_amount|validation_status|reason_code          |
+--------+---------+------------+------------+-----------------+---------------------+
|C001    |M001     |2026-01-15  |1200.0      |PASSED           |NULL                 |
|C002    |M002     |2026-02-10  |800.0       |PASSED           |NULL                 |
|C003    |M003     |2026-03-05  |500.0       |FAILED           |NO_ACTIVE_ELIGIBILITY|
|C004    |M004     |2026-01-20  |700.0       |FAILED           |NO_ACTIVE_ELIGIBILITY|
|C005    |M001     |2026-04-10  |300.0       |FAILED           |NO_ACTIVE_ELIGIBILITY|
+--------+---------+------------+------------+-----------------+---------------------+



**Step 10: Agent Summary Report**

In [12]:
final_silver_claims.groupBy(
    "validation_status", "reason_code"
).count().show()

+-----------------+--------------------+-----+
|validation_status|         reason_code|count|
+-----------------+--------------------+-----+
|           PASSED|                NULL|    2|
|           FAILED|NO_ACTIVE_ELIGIBI...|    3|
+-----------------+--------------------+-----+

